In [3]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
            # data_dft_d3bj = data_d3bj
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero

        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

print("Summary")
display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# # save summary to csv with date
# df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
# df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
# mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
# wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
# wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# # save summary to excel with date
# df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
# df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
# mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
# wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
# wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 45.39337691338733 kcal/mol, 128 in 1235076_W4_11
  -1 * W4_11-s4-c2v  -1 * -46.20945413445588  4 * W4_11-s  4 * -0.2040193052671384
Top 2 AI: 37.72718532665749 kcal/mol, 107 in 1235076_W4_11
  -1 * W4_11-so3  -1 * -37.68870474904543  1 * W4_11-s  1 * -0.2040193052671384  3 * W4_11-o  3 * 0.08083329429064179
Top 3 AI: 36.56782507803291 kcal/mol, 116 in 1235076_W4_11
  -1 * W4_11-p4  -1 * -35.85217151453253  4 * W4_11-p  4 * 0.1789133908750955
Top 4 AI: 34.95127727143699 kcal/mol, 131 in 1235076_W4_11
  -1 * W4_11-oclo  -1 * -33.724435767158866  2 * W4_11-o  2 * 0.08083329429064179  1 * W4_11-cl  1 * 1.0651749157113954
Top 5 AI: 30.338845555917942 kcal/mol, 135 in 1235076_W4_11
  -1 * W4_11-cloo  -1 * -29.112004051625263  1 * W4_11-cl  1 * 1.0651749157113954  2 * W4_11-o  2 * 0.08083329429064179
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.610892266675364 kcal/mol
Top 5 DFT: 55.139185982690

/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packa

data_path       1235076                                            \
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip   
summary        0.284712      0.264132      0.034112      0.033811   
W4_11          0.191172      0.148743      0.022156      0.021439   
G21EA          0.244469      0.109529      0.012193      0.012762   
G21IP          0.259469      0.102473      0.005239      0.006154   
DIPCS10        0.107399      0.107454      0.003474      0.003622   
PA26           0.285697      0.298218      0.036664      0.038429   
SIE4x4         0.083284         0.094       0.00212      0.001311   
ALKBDE10       0.217158      0.105669      0.066712      0.061711   
YBDE18         0.264223      0.276224      0.044874      0.041601   
AL2X6          0.411618       0.39636      0.001488      0.002336   
HEAVYSB11      0.383537      0.278584      0.012617      0.009043   
NBPRC            0.2487      0.252691      0.038168      0.032482   
ALK8            0.22259      0.187231      0.035284      0.040223   
RC21           0.312169      0.258996      0.085208      0.082447   
G2RC           0.169802      0.181458      0.015693      0.015617   
BH76RC         0.160843      0.140369      0.058352      0.057338   
FH51            0.31128      0.324197       0.02759      0.028992   
TAUT15         0.396621      0.413574      0.065406      0.069453   
DC13           0.402334      0.397878      0.018901      0.019322   
MB16_43        0.542858      0.525726      0.095966      0.086529   
DARC            0.41859      0.443827       0.02012      0.019082   
RSE43          0.206564      0.222802      0.034989      0.035941   
BSR36          0.463301      0.495755      0.001428      0.001683   
CDIE20         0.357728      0.360334      0.039291      0.043548   
ISO34          0.252668      0.265666      0.027807      0.028823   
PArel          0.505888      0.547394      0.071505      0.071661   
BH76           0.160843      0.140369      0.058352      0.057338   
BHPERI         0.232091      0.233558      0.039592      0.041322   
BHDIV10             NaN           NaN           NaN           NaN   
INV24               NaN           NaN           NaN           NaN   
BHROT27             NaN           NaN           NaN           NaN   
PX13                NaN           NaN           NaN           NaN   
WCPT18              NaN           NaN           NaN           NaN   
RG18                NaN           NaN           NaN           NaN   
ADIM6               NaN           NaN           NaN           NaN   
S22                 NaN           NaN           NaN           NaN   
S66                 NaN           NaN           NaN           NaN   
WATER27             NaN           NaN           NaN           NaN   
CARBHB12            NaN           NaN           NaN           NaN   
PNICO23             NaN           NaN           NaN           NaN   
HAL59               NaN           NaN           NaN           NaN   
AHB21               NaN           NaN           NaN           NaN   
CHB6                NaN           NaN           NaN           NaN   
IL16                NaN           NaN           NaN           NaN   
IDISP               NaN           NaN           NaN           NaN   
ICONF               NaN           NaN           NaN           NaN   
ACONF               NaN           NaN           NaN           NaN   
Amino20x4      0.624826      0.684993      0.033288      0.034358   
PCONF21             NaN           NaN           NaN           NaN   
MCONF               NaN           NaN           NaN           NaN   
SCONF               NaN           NaN           NaN           NaN   
BUT14DIOL           NaN           NaN           NaN           NaN   

data_path       1513512                                            
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip  
summary       31.939934     31.966861      0.837334      0.837474  
W4_11          9.982846      9.987136      0.558151      0.557822  
G

MAE


data_path   1235076                                                       \
Disp type        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       7.328837  13.945213  7.984829  14.550244  7.423161  13.998444   
sub2       8.388119   6.882858  8.169398   9.387391  6.750702     7.0235   
sub3       6.223826   8.574282  7.235274   9.609094  6.955988   9.322624   
sub4            NaN        NaN       NaN        NaN       NaN        NaN   
sub5       0.686326   0.611584  0.663138   0.659246  0.584206   0.594011   

data_path             1513512                                             \
Disp type Processed        AI        DFT    AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1        17 / 18  6.282354  13.717062   6.868466  14.265974  6.365582   
sub2          3 / 7  7.536765   6.612828  11.817093   9.131704  8.930029   
sub3          1 / 7  4.620451    6.26131   5.733397   7.356016  5.317836   
sub4         0 / 11   2.38953   3.272965   3.818374   5.029019  3.790518   
sub5          0 / 8  1.142385   1.242044    0.71893   0.945989  0.669339   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.720054      DONE  
sub2        6.760778      DONE  
sub3        6.935442      DONE  
sub4        4.986451      DONE  
sub5        0.892802     6 / 8

...

data_path    1235076                                                        \
Disp type         AI       DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        7.397286   7.40209   7.578915   7.417581   7.188357   7.108243   
sub2       13.267949  12.89935  11.811866  10.370841  11.652828  10.447234   
sub3        5.493693  6.587866   7.418119   8.566477   6.901178   8.007083   
sub4             NaN       NaN        NaN        NaN        NaN        NaN   
sub5        6.863265  6.115841   6.631379   6.592461   5.842065   5.940114   

data_path              1513512                                              \
Disp type Processed         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO   
sub1        17 / 18   5.917804   7.427913   6.173631   7.422353   5.786012   
sub2          3 / 7   9.677109  12.530346   7.731942   9.967392   7.407743   
sub3          1 / 7   5.362819   6.982875   6.557344   8.149361   5.935941   
sub4         0 / 11  10.472747  11.046974  12.507724  14.856703  11.478243   
sub5          0 / 8  10.571404  11.305352   6.432975   8.030856    5.87978   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        7.109349      DONE  
sub2       10.004803      DONE  
sub3        7.519994      DONE  
sub4       13.779798      DONE  
sub5        7.496872     6 / 8

wtmad_2


data_path    1235076                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        9.964339  11.601842  10.423415  11.906155  10.027065  11.567444   
sub2       10.635844   9.583686   7.383424    5.99319   7.892462    6.57137   
sub3        3.730553   5.140742   4.335414   5.759562   4.168377   5.588269   
sub4             0.0        0.0        0.0        0.0        0.0        0.0   
sub5        0.980795   0.873984   0.947657   0.942095    0.83486   0.848872   

data_path             1513512                                          \
Disp type Processed        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO   
sub1        17 / 18  2.771608  4.201338  2.916965  4.274734  2.778466   
sub2          3 / 7  3.330457  3.777436  2.126894  2.468684  2.260698   
sub3          1 / 7   2.06621  2.711546  2.515035  3.157212  2.344022   
sub4         0 / 11  4.289552  4.324256  6.316272  7.115216  6.077037   
sub5          0 / 8  3.981232  4.328172   2.72087  3.442019  2.427603   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        4.148409      DONE  
sub2        2.645745      DONE  
sub3        2.984382      DONE  
sub4        6.869745      DONE  
sub5        3.143916     6 / 8

Summary of Subset
MAE


data_path    1235076                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11      11.513239  29.506863  13.095673  31.211753  11.847678  29.861055   
G21EA       1.643243   9.754522   1.640386   9.751745   1.650025   9.757546   
G21IP       2.743693   8.953334   2.749886   8.946699   2.748434     8.9519   
DIPCS10     2.334857  12.310852   2.332461  12.334099   2.306808  12.264655   
PA26        1.977497   2.200766   2.019546   1.900152   1.962193    2.00824   
SIE4x4     19.458896  21.908516  19.862499  22.337814  19.890234  22.339928   
ALKBDE10   12.544776  18.125246   13.23338  18.838948  12.555049  18.135588   
YBDE18      6.241698   8.145393   6.129521   7.265227   5.593195   7.472761   
AL2X6       2.667034   5.659145    4.17442   1.332003   1.953157   1.829895   
HEAVYSB11   4.939612   5.391476   7.679056   8.010474   5.652736   5.984154   
NBPRC       7.876293    2.23255   5.143924   2.544088   5.796667   2.274234   
ALK8        6.994371   4.400063   4.110921   3.174803   4.572721   2.559066   
RC21        5.054487   4.820926   6.998495   6.671006   6.339175     6.0297   
G2RC        8.020576   5.917203   8.984188   6.933515   8.502842   6.417298   
BH76RC      3.595504   3.484561   3.759081   3.539828   3.631513   3.536982   
FH51        3.886731   3.364679   4.731923   3.448841   4.574835    3.36759   
TAUT15      3.626924    2.16453   3.661524   2.175485   3.640784   2.145937   
DC13       12.101761  13.090861  11.337676  12.474047  10.383917  11.475687   
MB16_43    21.937571  15.271927  27.107995  36.477768  19.543097  23.122795   
DARC        4.028026   10.75415   6.104753    3.43411    4.01005    5.57797   
RSE43       3.882376    3.15305   3.660374   2.931048   3.508301   2.778975   
BSR36      10.855678   8.400969   3.502366   1.213694   5.533815   3.079106   
CDIE20      1.070272   1.599507   0.824225   1.336544   0.833705   1.347484   
ISO34       1.764897   1.793521   1.838231   1.620593   1.747622   1.667745   
PArel       2.450681   2.029405   2.368092    2.00038   2.490393   1.922896   
BH76        6.419981   9.107946   7.186152     9.8892   6.970713   9.676053   
BHPERI      4.567404   4.067785   7.650085   7.243753   6.831644   6.338113   
BHDIV10            0          0          0          0          0          0   
INV24              0          0          0          0          0          0   
BHROT27            0          0          0          0          0          0   
PX13               0          0          0          0          0          0   
WCPT18             0          0          0          0          0          0   
RG18               0          0          0          0          0          0   
ADIM6              0          0          0          0          0          0   
S22                0          0          0          0          0          0   
S66                0          0          0          0          0          0   
WATER27            0          0          0          0          0          0   
CARBHB12           0          0          0          0          0          0   
PNICO23            0          0          0          0          0          0   
HAL59              0          0          0          0          0          0   
AHB21              0          0          0          0          0          0   
CHB6               0          0          0          0          0          0   
IL16               0          0          0          0          0          0   
IDISP              0          0          0          0          0          0   
ICONF              0          0          0          0          0          0   
ACONF              0          0          0          0          0          0   
Amino20x4   0.686326   0.611584   0.663138   0.659246   0.584206   0.594011   
PCONF21            0          0          0          0          0          0   
MCONF              0          0          0        